# Disambiguation with Google Gemini API

In this notebook, we explore entity disambiguation using Google Gemini's capabilities.

The prompts and settings of this notebook are designed to work with a previously annotated text in the format `[Entity](LABEL)`, which we can produce with a NER model.

Here we attempt disambiguation of labelled entities against WikiData, extracting location coordinates when available. We will try the following approaches:
1. Working with a single entity to disambiguate in a sample sentence.
2. Disambiguating all entities in a local txt file.
3. Disambiguating entities using a list of candidates.

## Installing Packages

We will need the following packages:
- `google.generativeai`
- `python-dotenv` (only necessary if working on your local device)
- `spaCy` and `pandas` for visualization and analysis of the results
- `pydantic` to constrain the output to a particular schema (optional)

In [ ]:
%pip install google-generativeai python-dotenv pydantic spacy pandas

### Load the Google Gemini API

**To work on Google Colab:**
1. Get your API key from [Google AI Studio](https://makersuite.google.com/app/apikey)
2. Import the API key in the secrets of this notebook
3. Import the necessary packages and the api key with
```python
from google.colab import userdata
userdata.get('YOUR_API_KEY_NAME')
```

**To work on your device:**
1. Get your API key from [Google AI Studio](https://makersuite.google.com/app/apikey)
2. Create a `.env` file in the same directory with: `GOOGLE_API_KEY=your_actual_api_key_here`
3. Replace `your_actual_api_key_here` with your real API key
4. Call the API key with:
```python
from dotenv import load_dotenv
load_dotenv()
GOOGLE_API_KEY = os.getenv('YOUR_API_KEY_NAME')
```

Calling the API differs slightly depending on whether you're working on your own devide or on this notebook.

In [ ]:
# uncomment if working on own device, to allow access to parent directory.
#import sys
#sys.path.append("..")

In [ ]:
import os
import re
import google.generativeai as genai
from pydantic import BaseModel
import json
import pandas as pd

import spacy
from spacy import displacy
nlp = spacy.load("en_core_web_sm")

# we import the necessary packages to call the API from this notebook's secrets.
from google.colab import userdata

In [ ]:
# necessary packages to call the API from this notebook's secrets.
#from google.colab import userdata
#api_key = userdata.get('GOOGLE_API_KEY')

# uncomment if working on own device, to allow loading the .env variable.

#from dotenv import load_dotenv
#load_dotenv()
#GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

### Configure the Google Gemini Client

The first time you do this, you may have to import your api key from Google AI Studio and grant access to this specific notebook.

In [ ]:
genai.configure(api_key=userdata.get('GOOGLE_API_KEY')) # if working on this colab notebook
#genai.configure(api_key=YOUR_API_KEY) # if working on own device
model = genai.GenerativeModel('gemini-2.5-flash') # here we define the model we want to work with

## Functions for this notebook

The functions in this section will help with formatting, visualizing, and converting the outputs of the response in various ways. Primarily, we'll work with the spaCy module `displaCy`, which enables a convenient visualization of the annotations, and with `pandas`, to process our output as a table.

- `annotated_text_to_spacy_doc()` is a convenience function to convert an annotated text with a format [Entity](LABEL) to a spaCy Doc with entity spans. This is required to process the text with displaCy.
- `visualize_annotated_text()` takes the output of this function and converts it into a format that displaCy can display.
- `parse_json_with_sources()` takes the response of our LLM (the disambiguated entities) and formats it as JSON.
- `output_ents_pandas(json_output)` takes input text as a spaCy Doc object and the JSON, and produces a list of entities in displaCy-compatible format, and a list of entities in pandas-compatible format. Note that you will need to process your input file as a spaCy doc first.

In [ ]:
from spacy.tokens import Doc, Span

def annotated_text_to_spacy_doc(text, nlp=None):
    """
    Converts annotated text in format [Entity](LABEL) to a spaCy Doc with entity spans.

    Args:
        text (str): Text with annotations like "[Tom](PERSON) worked for [Microsoft](ORGANIZATION)"
        nlp (spacy.Language, optional): spaCy language model. If None, uses blank English model.

    Returns:
        spacy.tokens.Doc: spaCy document with entity spans set

    Example usage:
        >>> text = "[Tom](PERSON) worked for [Microsoft](ORGANIZATION) in 2020 before he lived in [Rome](LOCATION)."
        >>> doc = annotated_text_to_spacy_doc(text)
        >>> spacy.displacy.render(doc, style="ent")
    """
    if nlp is None:
        nlp = spacy.blank("en")

    # Pattern to match [text](LABEL) format
    pattern = r'\[([^\]]+)\]\(([^)]+)\)'

    # Parse the text to extract tokens and entity information
    tokens = []
    entity_spans = []  # List of (start_token_idx, end_token_idx, label)
    custom_labels = set()

    # Split text by the pattern and process each part
    last_end = 0
    token_idx = 0

    for match in re.finditer(pattern, text):
        # Add tokens before the entity
        before_entity = text[last_end:match.start()]
        if before_entity.strip():
            # Tokenize the text before the entity
            before_tokens = before_entity.split()
            tokens.extend(before_tokens)
            token_idx += len(before_tokens)

        # Add the entity tokens
        entity_text = match.group(1)
        entity_label = match.group(2)
        custom_labels.add(entity_label)

        # Tokenize the entity text
        entity_tokens = entity_text.split()
        start_token_idx = token_idx
        tokens.extend(entity_tokens)
        token_idx += len(entity_tokens)
        end_token_idx = token_idx

        # Store entity span information
        entity_spans.append((start_token_idx, end_token_idx, entity_label))

        last_end = match.end()

    # Add any remaining tokens after the last entity
    remaining = text[last_end:]
    if remaining.strip():
        remaining_tokens = remaining.split()
        tokens.extend(remaining_tokens)

    # Add custom labels to the NLP model if they don't exist
    if "ner" not in nlp.pipe_names:
        ner = nlp.add_pipe("ner")
    else:
        ner = nlp.get_pipe("ner")

    for label in custom_labels:
        ner.add_label(label)

    # Create spaces array (True for tokens that should have a space after them)
    # Simple heuristic: all tokens except the last one get a space
    spaces = [True] * len(tokens)
    if tokens:
        spaces[-1] = False

    # Create the Doc from tokens
    doc = Doc(nlp.vocab, words=tokens, spaces=spaces)

    # Create entity spans
    entities = []
    for start_idx, end_idx, label in entity_spans:
        if start_idx < len(doc) and end_idx <= len(doc):
            span = Span(doc, start_idx, end_idx, label=label)
            entities.append(span)

    # Set entities on the document
    doc.ents = entities

    return doc


def visualize_annotated_text(text, nlp=None, style="ent", jupyter=True):
    """
    Convenience function to convert annotated text and visualize it with displaCy.

    Args:
        text (str): Text with annotations like "[Tom](PERSON) worked for [Microsoft](ORGANIZATION)"
        nlp (spacy.Language, optional): spaCy language model. If None, uses blank English model.
        style (str): displaCy style ("ent" or "dep")
        jupyter (bool): Whether to render for Jupyter notebook

    Returns:
        Rendered visualization (HTML string if not in Jupyter)
    """
    doc = annotated_text_to_spacy_doc(text, nlp)

    try:
        import spacy
        return spacy.displacy.render(doc, style=style, jupyter=jupyter)
    except ImportError:
        print("spaCy not installed. Please install with: pip install spacy")
        return None

In [ ]:
# parse the output text in json format
def parse_json_with_sources(text):
    # Find the start of the JSON data within a code block
    json_data = text.split("```json")[1]
    json_data, sources = json_data.split("```")
    # Load the JSON string into a Python dictionary
    json_data = json.loads(json_data)
    # Return the parsed JSON data and the extracted sources text
    return json_data, sources

In [ ]:
# format the output for visualization with pandas and displacy
def output_ents_pandas(doc, json_output):
    """
    Function to format the response output for visualization with pandas and displacy.

    Args:
        doc (Doc): the input text processed as a spacy doc object, with annotations in format [Entity](LABEL)
        json_output (list): the output of the model formatted as json

    Returns:
        output_ents: a list of entities in the format required by displacy
        pandas_output: a list of entities in the format required by pandas
    """
    output_ents = []
    pandas_output = []
    for ent in doc.ents:
        found=False
        for item in json_output:
            if item["entity_text"] == ent.text:
              # this is the ents dict that will allow displacy rendering
              output_ents.append({"start": ent.start_char, "end": ent.end_char, "label": f'{ent.label_} <a href="https://www.wikidata.org/wiki/{item["wikidata_id"]}">{item["wikidata_id"]}</a>'})
              # next, we create a lat and a long variable to store the numerical value of each geographical coordinate retrieved from wikidata
              latitude = item['geographic_coordinates'][0] if item['geographic_coordinates'] else None
              longitude = item['geographic_coordinates'][1] if item['geographic_coordinates'] else None
              # finally, we append all the desired items to the pandas_output list, so that we can convert them into a dataframe later
              pandas_output.append({"entity_text": item["entity_text"], "label": item["label"], "wikidata_id": item["wikidata_id"], "latitude": latitude, "longitude": longitude, "ent_start": ent.start_char, "ent_end": ent.end_char, "source": item["sources"]})
              found=True
        if found==False:
            output_ents.append({"start": ent.start_char, "end": ent.end_char, "label": ent.label_})
            pandas_output.append({"entity_text": ent.text, "label": ent.label_, "wikidata_id": None, "ent_start": ent.start_char, "ent_end": ent.end_char})

    return output_ents, pandas_output

## Working with a sample text

In the next section, we'll test the model with a sample text encoded as markdown annotations ```[Entity](LABEL)```. We will define the TEXT variable and one sample ENTITY for the model to identify.

For convenience in handling formats, we'll ask Gemini to return the results as a list of entities, even though this list should only return one item.

In [ ]:
TEXT = "They marched from [Alexandria](LOCATION) through [Memphis](LOCATION) via the [Nile](LOCATION) to [Thebes](LOCATION)."
ENTITY_TO_IDENTIFY = "Nile"

### Crafting the Prompt

In [ ]:
prompt_disambiguation_single_entity = """
Please identify this entity and find its Wikidata ID. Use your knowledge to search for the correct Wikidata entry.

Entity to identify: {entity}

Context: {text}

Please provide the Wikidata ID for this entity. Consider the context to determine which specific entity is being referenced (e.g., if "Memphis" appears in a historical context about ancient Egypt, it likely refers to the ancient Egyptian city).

Only return the JSON output, nothing else. Return a list of entities with the following schema:

class Entity(BaseModel):
    entity_text: str
    label: str
    wikidata_id: str
    sources: list[str]
    geographic_coordinates: list[float]

Do not include any other text, just the JSON.
"""

In [ ]:
# testing if the prompt works with the correct variables
formatted_prompt_single_entity = prompt_disambiguation_single_entity.format(entity=ENTITY_TO_IDENTIFY, text=TEXT)
print(formatted_prompt_single_entity)


Please identify this entity and find its Wikidata ID. Use your knowledge to search for the correct Wikidata entry.

Entity to identify: Nile

Context: They marched from [Alexandria](LOCATION) through [Memphis](LOCATION) via the [Nile](LOCATION) to [Thebes](LOCATION).

Please provide the Wikidata ID for this entity. Consider the context to determine which specific entity is being referenced (e.g., if "Memphis" appears in a historical context about ancient Egypt, it likely refers to the ancient Egyptian city).

Only return the JSON output, nothing else. Return a list of entities with the following schema:

class Entity(BaseModel):
    entity_text: str
    label: str
    wikidata_id: str
    sources: list[str]
    geographic_coordinates: list[float]

Do not include any other text, just the JSON.



### Calling Google Gemini

In [ ]:
response = model.generate_content(formatted_prompt_single_entity)
output_text_single_entity = response.text
print(output_text_single_entity)

```json
[
  {
    "entity_text": "Nile",
    "label": "Nile",
    "wikidata_id": "Q3392",
    "sources": [
      "Wikidata"
    ],
    "geographic_coordinates": [
      30.3444,
      30.5898
    ]
  }
]
```


### Parsing and visualizing the Output

Finally, we'll visualize the output using displacy.

- First, we'll load the input text as a spaCy doc using the default spacy nlp() module, and we'll convert it into a displacy-compatible format using the Visualization function ```annotated_text_to_spacy_doc()``` we defined above.
- Then, we'll incorporate the disambiguated entity in the displacy visualization.


In [ ]:
# if you skipped the required imports at the beginning, make sure to import displaCy and load the spacy model before you continue
#from spacy import displacy
#nlp = spacy.load("en_core_web_sm")

In [ ]:
doc = nlp(TEXT)
doc = annotated_text_to_spacy_doc(TEXT)

In [ ]:
displacy.render(doc, style="ent")

In [ ]:
# Parse the output text from the Gemini model
json_output_single_entity, sources = parse_json_with_sources(output_text_single_entity)
print(json_output_single_entity)

[{'entity_text': 'Nile', 'label': 'Nile', 'wikidata_id': 'Q3392', 'sources': ['Wikidata'], 'geographic_coordinates': [30.3444, 30.5898]}]


In [ ]:
# we parse the json output through our function output_ents_pandas() to produce entities in a format displacy can handle, and in a pandas format
output_ents_single_entity, pandas_output_single_entity = output_ents_pandas(doc, json_output_single_entity)

In [ ]:
# finally, we visualize our disambiguated entities in displacy.

dic_ents = {
    "text": doc.text,
    "ents": output_ents_single_entity,
    "title": None
}

displacy.render(dic_ents, manual=True, style="ent")

In [ ]:
# we can also load the results as a dataframe

df = pd.DataFrame(pandas_output_single_entity)
df.head()

,entity_text,label,wikidata_id,ent_start,ent_end,latitude,longitude,source
0,Alexandria,LOCATION,None,18,28,NaN,NaN,NaN
1,Memphis,LOCATION,None,37,44,NaN,NaN,NaN
2,Nile,Nile,Q3392,53,57,30.3444,30.5898,[Wikidata]
3,Thebes,LOCATION,None,61,67,NaN,NaN,NaN


In [ ]:
# optionally, we may want to save the output as a csv file
df.to_csv("/content/entities-disambiguation-web.csv", index=False)

## Disambiguate all entities in a local file

We may want to use a local file and disambiguate all entities present with a basic prompt like the one above. In this section, we'll open and read a file, and parse it with Gemini to retrieve a list of disambiguated entities that we can use afterwards for analysis and visualization.

Make sure you upload a .txt file in the correct markdown format: `[Entity](LABEL)`.

In [ ]:
# define the prompt to disambiguate all entities in a text.

prompt_all_entities = """
Disambiguate the entities in the following text.

{text}

Only return the JSON output, nothing else. Do so with the following schema:

Return a list of entities with the following schema:
class Entity(BaseModel):
    entity_text: str
    label: str
    wikidata_id: str
    sources: list[str]
    geographic_coordinates: list[float]

"""

In [ ]:
# open a local txt file and read it
with open("/content/input_text.txt", "r") as f:
  input_text = f.read()

In [ ]:
formatted_prompt_all_entities = prompt_all_entities.format(text=input_text)
print(formatted_prompt_all_entities)


Disambiguate the entities in the following text.

They marched from [Alexandria](LOCATION) through [Memphis](LOCATION) via the [Nile](LOCATION) to [Thebes](LOCATION).

[Alexander the Great](PERSON) conquered [Persia](LOCATION) and established the [Macedonian Empire](ORGANIZATION).

The [University of Oxford](ORGANIZATION) is located in [Oxford](LOCATION), [England](LOCATION).

[Albert Einstein](PERSON) developed the theory of relativity while working at the [Institute for Advanced Study](ORGANIZATION) in [Princeton](LOCATION).

Only return the JSON output, nothing else. Do so with the following schema:

Return a list of entities with the following schema:
class Entity(BaseModel):
    entity_text: str
    label: str
    wikidata_id: str
    sources: list[str]
    geographic_coordinates: list[float]




In [ ]:
# call the gemini model with the formatted prompt and process the sample text
response = model.generate_content(formatted_prompt_all_entities)
output_text_all_entities = response.text
print(output_text_all_entities)

```json
[
  {
    "entity_text": "Alexandria",
    "label": "LOCATION",
    "wikidata_id": "Q87",
    "sources": [
      "https://www.wikidata.org/wiki/Q87"
    ],
    "geographic_coordinates": [
      31.2,
      29.9167
    ]
  },
  {
    "entity_text": "Memphis",
    "label": "LOCATION",
    "wikidata_id": "Q142944",
    "sources": [
      "https://www.wikidata.org/wiki/Q142944"
    ],
    "geographic_coordinates": [
      29.8453,
      31.2581
    ]
  },
  {
    "entity_text": "Nile",
    "label": "LOCATION",
    "wikidata_id": "Q3390",
    "sources": [
      "https://www.wikidata.org/wiki/Q3390"
    ],
    "geographic_coordinates": [
      30.6667,
      31.1333
    ]
  },
  {
    "entity_text": "Thebes",
    "label": "LOCATION",
    "wikidata_id": "Q162453",
    "sources": [
      "https://www.wikidata.org/wiki/Q162453"
    ],
    "geographic_coordinates": [
      25.7,
      32.65
    ]
  },
  {
    "entity_text": "Alexander the Great",
    "label": "PERSON",
    "wikidata_id":

In [ ]:
# Parse the output text from the Gemini model
json_output_all_entities, sources = parse_json_with_sources(output_text_all_entities)
# Print the parsed JSON output
print(json_output_all_entities)

[{'entity_text': 'Alexandria', 'label': 'LOCATION', 'wikidata_id': 'Q87', 'sources': ['https://www.wikidata.org/wiki/Q87'], 'geographic_coordinates': [31.2, 29.9167]}, {'entity_text': 'Memphis', 'label': 'LOCATION', 'wikidata_id': 'Q142944', 'sources': ['https://www.wikidata.org/wiki/Q142944'], 'geographic_coordinates': [29.8453, 31.2581]}, {'entity_text': 'Nile', 'label': 'LOCATION', 'wikidata_id': 'Q3390', 'sources': ['https://www.wikidata.org/wiki/Q3390'], 'geographic_coordinates': [30.6667, 31.1333]}, {'entity_text': 'Thebes', 'label': 'LOCATION', 'wikidata_id': 'Q162453', 'sources': ['https://www.wikidata.org/wiki/Q162453'], 'geographic_coordinates': [25.7, 32.65]}, {'entity_text': 'Alexander the Great', 'label': 'PERSON', 'wikidata_id': 'Q8407', 'sources': ['https://www.wikidata.org/wiki/Q8407'], 'geographic_coordinates': []}, {'entity_text': 'Persia', 'label': 'LOCATION', 'wikidata_id': 'Q29427', 'sources': ['https://www.wikidata.org/wiki/Q29427'], 'geographic_coordinates': []},

In [ ]:
# format the input text as a spaCy Doc
doc = annotated_text_to_spacy_doc(input_text)
displacy.render(doc, style="ent")

In [ ]:
# parse the json output through our function output_ents_pandas() to produce entities in a format displacy can handle, and in a pandas format
output_ents_all_entities, pandas_output_all_entities = output_ents_pandas(doc, json_output_all_entities)

In [ ]:
dic_ents = {
    "text": doc.text,
    "ents": output_ents_all_entities,
    "title": None
}

displacy.render(dic_ents, manual=True, style="ent")

In [ ]:
# convert the results in a data frame and save to csv
df = pd.DataFrame(pandas_output_all_entities)
df

output = "/content/allentities.csv"
df.to_csv(output, index=False)

# print a preview of the first few rows of the dataframe
df.head()

,entity_text,label,wikidata_id,latitude,longitude,ent_start,ent_end,source
0,Alexandria,LOCATION,Q87,31.2000,29.9167,18,28,[https://www.wikidata.org/wiki/Q87]
1,Memphis,LOCATION,Q142944,29.8453,31.2581,37,44,[https://www.wikidata.org/wiki/Q142944]
2,Nile,LOCATION,Q3390,30.6667,31.1333,53,57,[https://www.wikidata.org/wiki/Q3390]
3,Thebes,LOCATION,Q162453,25.7000,32.6500,61,67,[https://www.wikidata.org/wiki/Q162453]
4,Alexander the Great,PERSON,Q8407,NaN,NaN,70,89,[https://www.wikidata.org/wiki/Q8407]


## Disambiguating entities using a list of candidates

Next, we'll try a matching approach by providing Gemini with a list of candidates for entities in a text.

In [ ]:
# if you are just starting from this section, remember to call the genai client again
#client = genai.GenerativeModel('gemini-2.5-flash')

### Define the prompt and the list of candidates

In [ ]:
# define the list of candidates
CANDIDATES = [{
  "text": "Monet",
  "label": "PERSON",
  "start_char": 22,
  "end_char": 27,
  "candidates": [
    {
      "id": "person/31450df4-cb6b-44f0-8335-38593ea70104",
      "type": "Person",
      "name": "Jean-Baptiste de Lamarck",
      "classifications": [
        {
          "id": "concept/6f652917-4c07-4d51-8209-fcdd4f285343",
          "type": "Type",
          "name": "male"
        },
        {
          "id": "concept/e46688bf-8720-4f67-85b2-d9e048b95506",
          "type": "Type",
          "name": "Naturalists"
        },
        {
          "id": "concept/b3a2d21c-2782-4da3-aaa4-53c444c4735e",
          "type": "Type",
          "name": "Biologists"
        },
        {
          "id": "concept/d799dcc0-7c99-494b-91c2-0ecc04fd8bc9",
          "type": "Type",
          "name": "Officers"
        },
        {
          "id": "concept/7e91736d-7107-4494-9695-542e76cbf320",
          "type": "Type",
          "name": "French"
        },
        {
          "id": "concept/b779de71-e499-43aa-abd3-ad991a0d1375",
          "type": "Type",
          "name": "Botanists"
        },
        {
          "id": "concept/0e64c455-7fd1-414a-89ce-38102f009ac4",
          "type": "Type",
          "name": "Zoologists"
        },
        {
          "id": "concept/49390038-5b23-441e-b8c5-b4b44d2c04a7",
          "type": "Type",
          "name": "Faculty"
        },
        {
          "id": "concept/787eed88-09dd-4961-99af-cd53378f3ce6",
          "type": "Type",
          "name": "Chemists"
        },
        {
          "id": "concept/4dbea3b6-9049-40bf-bc16-5b0a064ceb56",
          "type": "Type",
          "name": "Meteorologists"
        },
        {
          "id": "concept/9cb213a4-799a-4d64-b755-5980b3045a60",
          "type": "Type",
          "name": "Paleontologists"
        },
        {
          "id": "concept/62ba8667-022f-4c6f-88e2-d843f1462a08",
          "type": "Type",
          "name": "Malacologists"
        },
        {
          "id": "concept/50674beb-e61a-4f72-a34d-58e64f498bbc",
          "type": "Type",
          "name": "Encyclopedists"
        },
        {
          "id": "concept/51a2fcfd-d4b4-42af-872b-f8dcf4a62ced",
          "type": "Type",
          "name": "Authors"
        }
      ],
      "descriptions": [
        {
          "content": "Chevalier; Professor; franz\u00f6sischer Naturforscher, Biologe",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        },
        {
          "content": "French naturalist (1744-1829)",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        },
        {
          "content": "naturalista franc\u00e9s (1744-1829)",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        },
        {
          "content": "officier, naturaliste et professeur de zoologie fran\u00e7ais (1744-1829)",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        },
        {
          "content": "qu\u00edmico franc\u00eas",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        }
      ],
      "member_of": [
        {
          "id": "group/ae7678d4-6bf2-452c-8a7f-d170176fd5d3",
          "type": "Group",
          "name": "Soci\u00e9t\u00e9 philomathique de Paris"
        },
        {
          "id": "group/c2379f3d-47ad-4270-b905-380c91904a8d",
          "type": "Group",
          "name": "Acad\u00e9mie de Berlin"
        },
        {
          "id": "group/c50693b7-126c-4c3b-8f84-1bf21c662e65",
          "type": "Group",
          "name": "Bavarian Academy of Sciences and Humanities"
        }
      ],
      "birthDate": "1744-08-01T00:00:00",
      "birthPlace": {
        "id": "place/8940d47c-7650-4cef-b06e-46e30af65a04",
        "type": "Place",
        "name": "Bazentin"
      },
      "deathDate": "1829-12-18T00:00:00",
      "deathPlace": {
        "id": "place/8e117529-3872-494c-ab5f-8d7800be2c64",
        "type": "Place",
        "name": "Paris"
      }
    },
    {
      "id": "person/642a0152-1567-4fbe-93f3-66f11c5cab9a",
      "type": "Person",
      "name": "Claude Monet",
      "classifications": [
        {
          "id": "concept/7e91736d-7107-4494-9695-542e76cbf320",
          "type": "Type",
          "name": "French"
        },
        {
          "id": "concept/6f652917-4c07-4d51-8209-fcdd4f285343",
          "type": "Type",
          "name": "male"
        },
        {
          "id": "concept/0588f9d1-03e3-4b52-b2bf-dd41e601dcdc",
          "type": "Type",
          "name": "Artists"
        },
        {
          "id": "concept/98e4295b-7e89-4836-b601-a195888b6257",
          "type": "Type",
          "name": "caricaturists"
        },
        {
          "id": "concept/4f377430-c1ec-432d-b00c-d70264520e8e",
          "type": "Type",
          "name": "Landscape painters"
        },
        {
          "id": "concept/5272d911-5ccb-4a45-8571-1fed0176d361",
          "type": "Type",
          "name": "Painters"
        },
        {
          "id": "concept/b455d036-ded0-4b6a-b94a-d693dcd7dba4",
          "type": "Type",
          "name": "owners"
        },
        {
          "id": "concept/7ec0c9f8-b1ea-46d7-b5e6-36f23129db6c",
          "type": "Type",
          "name": "Impressionist artists"
        }
      ],
      "descriptions": [
        {
          "content": "French painter",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        },
        {
          "content": "French, 1840\u20131926",
          "classifications": [
            {
              "id": "concept/54e35d81-9548-4b4e-8973-de02b09bf9da",
              "type": "Type",
              "name": "display biography"
            }
          ]
        },
        {
          "content": "French painter, 1840-1926",
          "classifications": [
            {
              "id": "concept/54e35d81-9548-4b4e-8973-de02b09bf9da",
              "type": "Type",
              "name": "display biography"
            }
          ]
        },
        {
          "content": "He was a successful caricaturist in his native Le Havre, but after studying plein-air landscape painting, he moved to Paris in 1859. He soon met future Impressionists Camille Pissarro and Pierre-Auguste Renoir. Renoir and Monet began painting outdoors together in the late 1860s, laying the foundations of Impressionism. In 1874, with Pissarro and Edgar Degas, Monet helped organize the Soci\u00e9t\u00e9 Anonyme des Artistes, Peintres, Sculpteurs, Graveurs, etc., the formal name of the Impressionists' group. During the 1870s Monet developed his charateristic technique for rendering atmospheric outdoor light, using broken, rhythmic brushwork. Throughout his career, he remained loyal to the Impressionists' early goal of capturing the transitory effects of nature through direct observation. In 1890 he began creating paintings in series, depicting the same subject under various conditions and at different times of the day. His late pictures, made when he was half-blind, are shimmering pools of color almost totally devoid of form.",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        },
        {
          "content": "Peintre. - \u00c9tabli \u00e0 Giverny en 1883",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        }
      ],
      "birthDate": "1840-11-14T00:00:00",
      "birthPlace": {
        "id": "place/8e117529-3872-494c-ab5f-8d7800be2c64",
        "type": "Place",
        "name": "Paris"
      },
      "deathDate": "1926-12-05T00:00:00",
      "deathPlace": {
        "id": "place/1eead86b-4570-4217-b675-fb1fa81f2670",
        "type": "Place",
        "name": "Giverny"
      }
    },
    {
      "id": "person/bad186a1-bc28-4709-8edb-eca3a9faf387",
      "type": "Person",
      "name": "Monet, Jean, 1932-",
      "birthDate": "1932-01-01T00:00:00"
    },
    {
      "id": "person/39884fa6-b0e5-4fdf-98a7-1788f4bad5fb",
      "type": "Person",
      "name": "Monet, J.-C. (Jean-Claude)",
      "classifications": [
        {
          "id": "concept/7e91736d-7107-4494-9695-542e76cbf320",
          "type": "Type",
          "name": "French"
        }
      ],
      "birthDate": "1941-01-01T00:00:00"
    },
    {
      "id": "person/f368e56b-fe27-4f6f-9e16-75725afe8e31",
      "type": "Person",
      "name": "Carter, Frances Monet"
    },
    {
      "id": "person/a1dccf2f-48c7-43cb-a51c-cc2b4fa54958",
      "type": "Person",
      "name": "Monet, Paul",
      "classifications": [
        {
          "id": "concept/6f652917-4c07-4d51-8209-fcdd4f285343",
          "type": "Type",
          "name": "male"
        },
        {
          "id": "concept/d799dcc0-7c99-494b-91c2-0ecc04fd8bc9",
          "type": "Type",
          "name": "Officers"
        },
        {
          "id": "concept/7e91736d-7107-4494-9695-542e76cbf320",
          "type": "Type",
          "name": "French"
        }
      ],
      "descriptions": [
        {
          "content": "Franz\u00f6sischer Offizier der Ehrenlegion, Kapit\u00e4n einer kolonialen Artillerie",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        }
      ],
      "member_of": [
        {
          "id": "group/f6827e5b-8e0f-413c-ada1-4c4e0fc594d6",
          "type": "Group",
          "name": "Minist\u00e8re des colonies"
        },
        {
          "id": "group/a9170801-fd5d-4cbe-b55e-843465f806ab",
          "type": "Group",
          "name": "Acad\u00e9mie Goncourt"
        },
        {
          "id": "group/9064c768-7424-4ccf-9fca-4d8d357ce73a",
          "type": "Group",
          "name": "Vi\u1ec7t Nam Thanh Ni\u00ean H\u1ed9i"
        }
      ],
      "birthDate": "1884-01-13T00:00:00",
      "birthPlace": {
        "id": "place/8de9ae57-d9e0-44f4-8950-54442e0506a1",
        "type": "Place",
        "name": "Angers"
      },
      "deathDate": "1941-05-26T00:00:00"
    },
    {
      "id": "person/2c8940dc-46bb-4029-a2f6-65fd24e8be8c",
      "type": "Person",
      "name": "Laurette Alexis-Monet",
      "classifications": [
        {
          "id": "concept/a309a746-9e51-4c34-b207-7f4773d2ac1a",
          "type": "Type",
          "name": "female"
        },
        {
          "id": "concept/7e91736d-7107-4494-9695-542e76cbf320",
          "type": "Type",
          "name": "French"
        }
      ],
      "birthDate": "1923-07-10T00:00:00",
      "deathDate": "2011-12-15T00:00:00"
    },
    {
      "id": "person/39b693f8-d2da-4d4e-a7a7-fb3dfd8d769d",
      "type": "Person",
      "name": "Monet, Alice K. B.",
      "classifications": [
        {
          "id": "concept/a309a746-9e51-4c34-b207-7f4773d2ac1a",
          "type": "Type",
          "name": "female"
        }
      ]
    },
    {
      "id": "person/60391131-8295-487a-b786-1216c9cc63ef",
      "type": "Person",
      "name": "Monet-Viera, Molly"
    },
    {
      "id": "person/8b0a5321-663f-4681-a034-a57cf47e9383",
      "type": "Person",
      "name": "Monet, Chantal",
      "classifications": [
        {
          "id": "concept/a309a746-9e51-4c34-b207-7f4773d2ac1a",
          "type": "Type",
          "name": "female"
        },
        {
          "id": "concept/303558a7-ab8f-4b09-a7f7-fffc993a84f5",
          "type": "Type",
          "name": "Journalists"
        },
        {
          "id": "concept/83155191-338b-4396-90ee-f9a625bcbfd3",
          "type": "Type",
          "name": "Belgian"
        }
      ]
    },
    {
      "id": "Q296",
      "type": "Person",
      "name": "Claude Monet",
      "classifications": [
        {
          "id": "Q1028181",
          "type": "Type",
          "name": "pintor"
        },
        {
          "id": "Q1925963",
          "type": "Type",
          "name": "artista gr\u00e1fico"
        }
      ],
      "descriptions": [
        {
          "content": "French painter (1840\u20131926)",
          "classifications": []
        },
        {
          "content": "pintor franc\u00e9s",
          "classifications": []
        },
        {
          "content": "peintre impressionniste fran\u00e7ais",
          "classifications": []
        },
        {
          "content": "pintor franc\u00eas (1840-1926)",
          "classifications": []
        },
        {
          "content": "franz\u00f6sischer Maler des Impressionismus (1840\u20131926)",
          "classifications": []
        }
      ],
      "birthDate": "1840-11-14T00:00:00",
      "birthPlace": {
        "id": "Q90",
        "type": "Place",
        "name": "Paris"
      },
      "deathDate": "1926-12-05T00:00:00",
      "deathPlace": {
        "id": "Q165061",
        "type": "Place",
        "name": "Giverny"
      }
    },
    {
      "id": "Q24698278",
      "type": "Person",
      "name": "Monet",
      "descriptions": [
        {
          "content": "family name",
          "classifications": []
        },
        {
          "content": "apellido",
          "classifications": []
        },
        {
          "content": "nom de famille",
          "classifications": []
        },
        {
          "content": "sobrenome",
          "classifications": []
        },
        {
          "content": "Familienname",
          "classifications": []
        }
      ]
    },
    {
      "id": "Q2959838",
      "type": "Person",
      "name": "Charles Monnet",
      "classifications": [
        {
          "id": "Q1028181",
          "type": "Type",
          "name": "pintor"
        }
      ],
      "descriptions": [
        {
          "content": "French court painter (1732-1808)",
          "classifications": []
        },
        {
          "content": "pintor franc\u00e9s",
          "classifications": []
        },
        {
          "content": "peintre fran\u00e7ais",
          "classifications": []
        },
        {
          "content": "pintor franc\u00eas",
          "classifications": []
        },
        {
          "content": "franz\u00f6sischer Hofmaler",
          "classifications": []
        }
      ],
      "birthDate": "1732-01-10T00:00:00",
      "birthPlace": {
        "id": "Q90",
        "type": "Place",
        "name": "Paris"
      },
      "deathDate": "1819-03-19T00:00:00",
      "deathPlace": {
        "id": "Q90",
        "type": "Place",
        "name": "Paris"
      }
    },
    {
      "id": "Q8142",
      "type": "Person",
      "name": "\u901a\u8ca8",
      "descriptions": [
        {
          "content": "generally accepted medium of exchange for goods or services",
          "classifications": []
        },
        {
          "content": "medio de cambio utilizado para bienes o servicios",
          "classifications": []
        },
        {
          "content": "instrument de paiement en vigueur en un lieu et \u00e0 une \u00e9poque donn\u00e9e",
          "classifications": []
        },
        {
          "content": "unidade monet\u00e1ria, meio de pagamento",
          "classifications": []
        },
        {
          "content": "Verfassung und Ordnung des gesamten Geldwesens eines Staates",
          "classifications": []
        }
      ]
    },
    {
      "id": "Q119729672",
      "type": "Person",
      "name": "Monet",
      "descriptions": [
        {
          "content": "given name",
          "classifications": []
        }
      ]
    },
    {
      "id": "Q234900",
      "type": "Person",
      "name": "Linda Darnell",
      "classifications": [
        {
          "id": "Q2259451",
          "type": "Type",
          "name": "stage actor"
        },
        {
          "id": "Q10798782",
          "type": "Type",
          "name": "television actor"
        },
        {
          "id": "Q10800557",
          "type": "Type",
          "name": "film actor"
        }
      ],
      "descriptions": [
        {
          "content": "American actress (1923\u20131965)",
          "classifications": []
        },
        {
          "content": "actriz estadounidense",
          "classifications": []
        },
        {
          "content": "actrice am\u00e9ricaine",
          "classifications": []
        },
        {
          "content": "US-amerikanische Schauspielerin",
          "classifications": []
        },
        {
          "content": "Amerikaans actrice (1923\u20131965)",
          "classifications": []
        }
      ],
      "birthDate": "1923-10-16T00:00:00",
      "birthPlace": {
        "id": "Q16557",
        "type": "Place",
        "name": "Dallas"
      },
      "deathDate": "1965-04-10T00:00:00",
      "deathPlace": {
        "id": "Q1531184",
        "type": "Place",
        "name": "Glenview"
      }
    },
    {
      "id": "Q223162",
      "type": "Person",
      "name": "Mon\u00e9teau",
      "descriptions": [
        {
          "content": "commune in Yonne, France",
          "classifications": []
        },
        {
          "content": "comuna francesa",
          "classifications": []
        },
        {
          "content": "commune fran\u00e7aise du d\u00e9partement de l'Yonne",
          "classifications": []
        },
        {
          "content": "comuna francesa",
          "classifications": []
        },
        {
          "content": "franz\u00f6sische Gemeinde",
          "classifications": []
        }
      ]
    }
  ]
},
{
  "text": "Argenteuil",
  "label": "LOCATION",
  "start_char": 121,
  "end_char": 131,
  "candidates": [
    {
      "id": "place/4699255d-458a-4795-8b04-2614f1c171db",
      "type": "Place",
      "name": "Argenteuil",
      "part_of": [
        {
          "id": "place/b7e88db4-e572-46e6-9617-8a2594bcfa8c",
          "type": "Place",
          "name": "Argenteuil"
        }
      ]
    },
    {
      "id": "place/b4b825fd-4b8e-4642-b37b-0d076a5ccf74",
      "type": "Place",
      "name": "Argenteuil",
      "part_of": [
        {
          "id": "place/682402f8-cdc4-4ebc-ae38-5b3824d2e4aa",
          "type": "Place",
          "name": "Quebec"
        }
      ]
    },
    {
      "id": "place/2f05fdc5-7e9e-4936-bde8-84a88347fde7",
      "type": "Place",
      "name": "Argenteuil",
      "descriptions": [
        {
          "content": "regional county municipality in Quebec, Canada",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        },
        {
          "content": "municipalit\u00e9 r\u00e9gionale de comt\u00e9 du Qu\u00e9bec (Canada)",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        }
      ],
      "part_of": [
        {
          "id": "place/cd467ccf-665a-423f-a1b5-1785869d960f",
          "type": "Place",
          "name": "Laurentides"
        }
      ]
    },
    {
      "id": "place/bae1a4f6-a9f0-4bb6-83fc-faec8611194a",
      "type": "Place",
      "name": "arrondissement of Argenteuil",
      "descriptions": [
        {
          "content": "arrondissement of France",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        },
        {
          "content": "distrito de Francia",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        },
        {
          "content": "arrondissement fran\u00e7ais",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        },
        {
          "content": "Verwaltungseinheit in Frankreich",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        },
        {
          "content": "arrondissement in Val-d'Oise, Frankrijk",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        }
      ],
      "part_of": [
        {
          "id": "place/bb803f08-8018-4a00-814a-6fceb3ec6d28",
          "type": "Place",
          "name": "Essonne"
        }
      ]
    },
    {
      "id": "place/b7e88db4-e572-46e6-9617-8a2594bcfa8c",
      "type": "Place",
      "name": "Argenteuil",
      "descriptions": [
        {
          "content": "Argenteuil is a commune in the Val-d'Oise department in the \u00cele-de-France region, located about 15 kilometers northwest of Paris, France. (AI generated)",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        },
        {
          "content": "Stadt im nordwestlichen Vorortbereich von Paris, an der Seine, im D\u00e9partement Val d'Oise, Frankreich",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        },
        {
          "content": "commune in Val-d'Oise, France",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        },
        {
          "content": "comuna francesa",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        },
        {
          "content": "commune fran\u00e7aise du d\u00e9partement du Val-d'Oise",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        }
      ],
      "part_of": [
        {
          "id": "place/6271aae3-a32f-4aaa-883d-9c99a803a09c",
          "type": "Place",
          "name": "France"
        },
        {
          "id": "place/bae1a4f6-a9f0-4bb6-83fc-faec8611194a",
          "type": "Place",
          "name": "arrondissement of Argenteuil"
        },
        {
          "id": "place/bb803f08-8018-4a00-814a-6fceb3ec6d28",
          "type": "Place",
          "name": "Essonne"
        },
        {
          "id": "place/67b2f4c7-5915-483e-af7e-8e7c218e1b53",
          "type": "Place",
          "name": "Grand Paris"
        },
        {
          "id": "place/4699255d-458a-4795-8b04-2614f1c171db",
          "type": "Place",
          "name": "Argenteuil"
        },
        {
          "id": "place/c4067590-40a9-462c-995a-9c58f100e6e6",
          "type": "Place",
          "name": "Argenteuil"
        }
      ]
    },
    {
      "id": "place/4f8c46e0-3701-4871-8073-0116e17eeed1",
      "type": "Place",
      "name": "Saint-Andr\u00e9-d'Argenteuil",
      "descriptions": [
        {
          "content": "municipality in Quebec, Canada",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        },
        {
          "content": "municipio en la\u00a0provincia\u00a0de\u00a0Quebec,\u00a0Canad\u00e1",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        },
        {
          "content": "municipalit\u00e9 au Qu\u00e9bec (Canada)",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        }
      ],
      "part_of": [
        {
          "id": "place/123bf43c-269e-40fd-b37d-c564dce9ce9b",
          "type": "Place",
          "name": "Qu\u00e9bec"
        },
        {
          "id": "place/2f05fdc5-7e9e-4936-bde8-84a88347fde7",
          "type": "Place",
          "name": "Argenteuil"
        },
        {
          "id": "place/b4b825fd-4b8e-4642-b37b-0d076a5ccf74",
          "type": "Place",
          "name": "Argenteuil"
        },
        {
          "id": "place/cd467ccf-665a-423f-a1b5-1785869d960f",
          "type": "Place",
          "name": "Laurentides"
        }
      ]
    },
    {
      "id": "place/1a1b5be6-9f94-4a05-88af-1d49ea123f3c",
      "type": "Place",
      "name": "Argenteuil (Qu\u00e9bec : Division de recensement)"
    },
    {
      "id": "place/b1a8acf8-392b-4739-af32-8db989d806d0",
      "type": "Place",
      "name": "Argenteuil (Qu\u00e9bec)"
    },
    {
      "id": "place/afacb2bd-8041-4d59-a700-18009fae3ad1",
      "type": "Place",
      "name": "North River (Argenteuil, Qu\u00e9bec)"
    },
    {
      "id": "place/9a2c2a4c-dd02-4e54-907a-3b6208174a06",
      "type": "Place",
      "name": "Argenteuil",
      "classifications": [
        {
          "id": "concept/4c4443fb-d094-4de4-a5cb-5e3078d58f06",
          "type": "Type",
          "name": "Cities and towns"
        }
      ],
      "descriptions": [
        {
          "content": "Silver deposits here were exploited by Gauls; town was destroyed by Normans, but rebuilt; convent here was endowed by Charlemagne; was famous in the 12th century for abbess H\u00e9lo\u00efse, of the tragic H\u00e9lo\u00efse-Abelard romance; currently a residential area.",
          "classifications": [
            {
              "id": "concept/b9d84f17-662e-46ef-ab8b-7499717f8337",
              "type": "Type",
              "name": "descriptive note"
            }
          ]
        }
      ],
      "part_of": [
        {
          "id": "place/d5aeace4-86fa-4193-a508-4fa6c615432d",
          "type": "Place",
          "name": "\u00cele-de-France"
        }
      ]
    },
    {
      "id": "Q181946",
      "type": "Place",
      "name": "Argenteuil",
      "descriptions": [
        {
          "content": "commune in Val-d'Oise, France",
          "classifications": []
        },
        {
          "content": "comuna francesa",
          "classifications": []
        },
        {
          "content": "commune fran\u00e7aise du d\u00e9partement du Val-d'Oise",
          "classifications": []
        },
        {
          "content": "comuna francesa",
          "classifications": []
        },
        {
          "content": "franz\u00f6sische Gemeinde",
          "classifications": []
        }
      ],
      "part_of": [
        {
          "id": "Q511613",
          "type": "Place",
          "name": "arrondissement of Argenteuil"
        },
        {
          "id": "Q12784",
          "type": "Place",
          "name": "Val-d'Oise"
        },
        {
          "id": "Q16665915",
          "type": "Place",
          "name": "M\u00e9tropole du Grand Paris"
        }
      ]
    },
    {
      "id": "Q645211",
      "type": "Place",
      "name": "Argenteuil",
      "descriptions": [
        {
          "content": "regional county municipality in Quebec, Canada",
          "classifications": []
        },
        {
          "content": "municipalit\u00e9 r\u00e9gionale de comt\u00e9 du Qu\u00e9bec (Canada)",
          "classifications": []
        }
      ],
      "part_of": [
        {
          "id": "Q2304022",
          "type": "Place",
          "name": "Laurentides"
        }
      ]
    },
    {
      "id": "Q1151230",
      "type": "Place",
      "name": "Argenteuil-sur-Arman\u00e7on",
      "descriptions": [
        {
          "content": "commune in Yonne, France",
          "classifications": []
        },
        {
          "content": "comuna francesa",
          "classifications": []
        },
        {
          "content": "commune fran\u00e7aise du d\u00e9partement de l'Yonne",
          "classifications": []
        },
        {
          "content": "comuna francesa",
          "classifications": []
        },
        {
          "content": "franz\u00f6sische Gemeinde",
          "classifications": []
        }
      ],
      "part_of": [
        {
          "id": "Q1724141",
          "type": "Place",
          "name": "canton of Ancy-le-Franc"
        },
        {
          "id": "Q12816",
          "type": "Place",
          "name": "Yonne"
        },
        {
          "id": "Q700536",
          "type": "Place",
          "name": "arrondissement of Avallon"
        }
      ]
    },
    {
      "id": "Q2860941",
      "type": "Place",
      "name": "Argenteuil",
      "descriptions": [
        {
          "content": "provincial electoral district in Quebec, Canada",
          "classifications": []
        },
        {
          "content": "circonscription electorale provinciale du Qu\u00e9bec, Canada",
          "classifications": []
        },
        {
          "content": "Provinzwahlkreis in Qu\u00e9bec",
          "classifications": []
        }
      ],
      "part_of": [
        {
          "id": "Q176",
          "type": "Place",
          "name": "Quebec"
        }
      ]
    },
    {
      "id": "Q3095674",
      "type": "Place",
      "name": "Argenteuil",
      "descriptions": [
        {
          "content": "railway station in Argenteuil, France",
          "classifications": []
        },
        {
          "content": "estaci\u00f3n de tren en Francia",
          "classifications": []
        },
        {
          "content": "gare ferroviaire fran\u00e7aise",
          "classifications": []
        },
        {
          "content": "Bahnhof in Frankreich",
          "classifications": []
        },
        {
          "content": "spoorwegstation in Frankrijk",
          "classifications": []
        }
      ],
      "part_of": [
        {
          "id": "Q181946",
          "type": "Place",
          "name": "Argenteuil"
        }
      ]
    },
    {
      "id": "Q2860945",
      "type": "Place",
      "name": "Argenteuil",
      "descriptions": [
        {
          "content": "painting by \u00c9douard Manet, 1874",
          "classifications": []
        },
        {
          "content": "cuadro de \u00c9douard Manet",
          "classifications": []
        },
        {
          "content": "tableau d'\u00c9douard Manet",
          "classifications": []
        },
        {
          "content": "pintura de \u00c9douard Manet",
          "classifications": []
        },
        {
          "content": "Gem\u00e4lde von \u00c9douard Manet aus dem Jahr 1874",
          "classifications": []
        }
      ]
    },
    {
      "id": "Q20188741",
      "type": "Place",
      "name": "Argenteuil",
      "descriptions": [
        {
          "content": "painting by Claude Monet (c. 1872, National Gallery of Art)",
          "classifications": []
        },
        {
          "content": "cuadro de Claude Monet",
          "classifications": []
        },
        {
          "content": "peinture de Claude Monet (v. 1872, National Gallery of Art)",
          "classifications": []
        },
        {
          "content": "pintura de Claude Monet",
          "classifications": []
        },
        {
          "content": "\u00d6lgem\u00e4lde von Claude Monet",
          "classifications": []
        }
      ]
    }
  ]
}
]

In [ ]:
# define the input text
TEXT = "This painting depicts [Monet](PERSON)'s first wife, [Camille](PERSON), outside on a snowy day passing by the [French](LOCATION) doors of their home at [Argenteuil](LOCATION). Her face is rendered in a radically bold Impressionist technique of mere daubs of paint quickly applied, just as the snow and trees are defined by broad, broken strokes of pure white and green."

In [ ]:
# define the prompt
prompt_candidates = """
Disambiguate the entities in the following text.

{text}

Here are the Candidates:

{candidates}

Only return the JSON output, nothing else. Do so with the following schema:

Return a list of entities with the following schema:

class Entity(BaseModel):
    entity_text: str
    label: str
    wikidata_id: str
    sources: list[str]
    geographic_coordinates: list[float]

"""

In [ ]:
# test if the prompt works with the specified input data
formatted_prompt_candidates = prompt_candidates.format(text=TEXT, candidates=CANDIDATES)
print(formatted_prompt_candidates)


Disambiguate the entities in the following text.

This painting depicts [Monet](PERSON)'s first wife, [Camille](PERSON), outside on a snowy day passing by the [French](LOCATION) doors of their home at [Argenteuil](LOCATION). Her face is rendered in a radically bold Impressionist technique of mere daubs of paint quickly applied, just as the snow and trees are defined by broad, broken strokes of pure white and green.

Here are the Candidates:

[{'text': 'Monet', 'label': 'PERSON', 'start_char': 22, 'end_char': 27, 'candidates': [{'id': 'person/31450df4-cb6b-44f0-8335-38593ea70104', 'type': 'Person', 'name': 'Jean-Baptiste de Lamarck', 'classifications': [{'id': 'concept/6f652917-4c07-4d51-8209-fcdd4f285343', 'type': 'Type', 'name': 'male'}, {'id': 'concept/e46688bf-8720-4f67-85b2-d9e048b95506', 'type': 'Type', 'name': 'Naturalists'}, {'id': 'concept/b3a2d21c-2782-4da3-aaa4-53c444c4735e', 'type': 'Type', 'name': 'Biologists'}, {'id': 'concept/d799dcc0-7c99-494b-91c2-0ecc04fd8bc9', 'type'

### Visualize the disambiguated entities

In [ ]:
# call the gemini client and run the disambiguation task with the prompt
response = model.generate_content(formatted_prompt_candidates)
output_text_candidates = response.text
print(output_text_candidates)

```json
[
  {
    "entity_text": "Monet",
    "label": "PERSON",
    "wikidata_id": "Q296",
    "sources": [
      "https://www.wikidata.org/wiki/Q296"
    ],
    "geographic_coordinates": []
  },
  {
    "entity_text": "Camille",
    "label": "PERSON",
    "wikidata_id": "Q2726759",
    "sources": [
      "https://www.wikidata.org/wiki/Q2726759"
    ],
    "geographic_coordinates": []
  },
  {
    "entity_text": "French",
    "label": "LOCATION",
    "wikidata_id": "Q142",
    "sources": [
      "https://www.wikidata.org/wiki/Q142"
    ],
    "geographic_coordinates": [
      47.0,
      2.0
    ]
  },
  {
    "entity_text": "Argenteuil",
    "label": "LOCATION",
    "wikidata_id": "Q181946",
    "sources": [
      "https://www.wikidata.org/wiki/Q181946"
    ],
    "geographic_coordinates": [
      48.94638889,
      2.24
    ]
  }
]
```


In [ ]:
# we call the function parse_json_with_sources to parse the results as json

json_output_candidates, sources = parse_json_with_sources(output_text_candidates)
print(json_output_candidates)

[{'entity_text': 'Monet', 'label': 'PERSON', 'wikidata_id': 'Q296', 'sources': ['https://www.wikidata.org/wiki/Q296'], 'geographic_coordinates': []}, {'entity_text': 'Camille', 'label': 'PERSON', 'wikidata_id': 'Q2726759', 'sources': ['https://www.wikidata.org/wiki/Q2726759'], 'geographic_coordinates': []}, {'entity_text': 'French', 'label': 'LOCATION', 'wikidata_id': 'Q142', 'sources': ['https://www.wikidata.org/wiki/Q142'], 'geographic_coordinates': [47.0, 2.0]}, {'entity_text': 'Argenteuil', 'label': 'LOCATION', 'wikidata_id': 'Q181946', 'sources': ['https://www.wikidata.org/wiki/Q181946'], 'geographic_coordinates': [48.94638889, 2.24]}]


In [ ]:
doc = annotated_text_to_spacy_doc(TEXT)
displacy.render(doc, style="ent")

In [ ]:
output_ents_candidates, pandas_output_candidates = output_ents_pandas(doc, json_output_candidates)

In [ ]:
dic_ents = {
    "text": doc.text,
    "ents": output_ents_candidates,
    "title": None
}

displacy.render(dic_ents, manual=True, style="ent")

In [ ]:
# convert the results in a data frame and save to csv
df = pd.DataFrame(pandas_output_candidates)
df

output = "/content/allentitiesfromcandidates.csv"
df.to_csv(output, index=False)
# print a preview of the first few rows of the dataframe
df.head()

,entity_text,label,wikidata_id,latitude,longitude,ent_start,ent_end,source
0,Monet,PERSON,Q296,NaN,NaN,22,27,[https://www.wikidata.org/wiki/Q296]
1,Camille,PERSON,Q2726759,NaN,NaN,43,50,[https://www.wikidata.org/wiki/Q2726759]
2,French,LOCATION,Q142,47.000000,2.00,91,97,[https://www.wikidata.org/wiki/Q142]
3,Argenteuil,LOCATION,Q181946,48.946389,2.24,121,131,[https://www.wikidata.org/wiki/Q181946]
